In [1]:

import pandas as pd
import os

import random

from collections import Counter
from typing import Dict, List

# Type alias for clarity: each fold is a dict with three lists of patient IDs (str)
Fold = Dict[str, List[str]]      # {'train': [...], 'val': [...], 'test': [...]}

In [2]:
# Functions -------------------------
def make_folds_greedy(patient_measurements: dict, k=5, seed=42):
    """
    patient_measurements : dict {patient_id: n_meas}
    Devuelve una lista de k listas (IDs de pacientes por fold-test),
    con los totales de mediciones lo más iguales posible.
    """
    rng = random.Random(seed)

    # Ordena por nº de mediciones (desc) y baraja un poco para romper empates
    items = list(patient_measurements.items())
    rng.shuffle(items)
    items.sort(key=lambda x: x[1], reverse=True)

    folds = [[] for _ in range(k)]
    fold_load = [0] * k                       # mediciones acumuladas

    for pid, n in items:
        idx = fold_load.index(min(fold_load)) # fold más “ligero”
        folds[idx].append(pid)
        fold_load[idx] += n                   # actualiza su peso

    return folds

def make_folds_greedy_splits(   patient_measurements: Dict[str, int],   # ← IDs como str
                                k: int = 5,
                                val_frac: float = 0.10,
                                seed: int = 42) -> List[Fold]:
    """
    Reparte pacientes (clave str) y su nº de mediciones (valor int)
    en k folds balanceando la carga total.
    Devuelve: list[ {'train': [...], 'val': [...], 'test': [...]} ]
    """
    rng = random.Random(seed)

    # 1) asignar pacientes grandes primero al fold-test más ligero
    items = list(patient_measurements.items())   # [(pid, n_meas), ...]
    rng.shuffle(items)
    items.sort(key=lambda x: x[1], reverse=True)

    test_folds: List[List[str]] = [[] for _ in range(k)]
    fold_load  = [0] * k                         # suma de mediciones por fold

    for pid, n in items:
        idx = fold_load.index(min(fold_load))
        test_folds[idx].append(pid)
        fold_load[idx] += n

    # 2) dentro de cada fold-test, crear train y val a partir del resto
    all_ids = list(patient_measurements.keys())
    folds: List[Fold] = []

    for i in range(k):
        test_ids = set(test_folds[i])
        remaining = [pid for pid in all_ids if pid not in test_ids]

        rng.shuffle(remaining)
        cut = int(val_frac * len(remaining))     # 20 % para validación
        val_ids   = remaining[:cut]
        train_ids = remaining[cut:]

        folds.append({
            'train': train_ids,
            'val'  : val_ids,
            'test' : list(test_ids)
        })

    return folds

def validate_folds(
        folds: List[Fold],
        patient_measurements: Dict[str, int],
        tolerance_pct: float = 10.0,
        horizon: int = 2) -> None:
    """
    Validate a list of patient-wise cross-validation folds.

    Parameters
    ----------
    folds : list[Fold]
        Output of `make_folds_greedy_splits` or similar.  Each element is a
        dictionary with keys ``'train'``, ``'val'``, ``'test'`` mapping to lists
        of patient IDs **(strings)**.

    patient_measurements : dict[str, int]
        Mapping ``{patient_id: number_of_measurements}``.  Used to check load
        balancing (Rule R4).

    tolerance_pct : float, default 10.0
        Maximum allowed percentage deviation from the mean test load before
        a warning is raised (Rule R4).
    
    horizon : int, default 2
        The horizon used for the folds, included in the printed messages.

    Raises
    ------
    AssertionError
        If any of the mandatory rules (R1–R3) is violated.

    Side effects
    ------------
    * Prints a success message for each rule that passes.
    * Prints the test-set load per fold and the percentage deviation from the
      mean (always), and a warning if any fold exceeds *tolerance_pct*.

    Notes
    -----
    *R4* (load balance) is **informative**; it does **not** raise an error.  
    Adjust *tolerance_pct* as appropriate for your project.  
    """
    
    print(f"\nValidating folds for horizon: {horizon}...")
    
    all_patients = set(patient_measurements)

    # ------------------------------------------------------------------ R1/R3
    for idx, fold in enumerate(folds):
        tr, va, te = map(set, (fold['train'], fold['val'], fold['test']))

        # R1 – exclusivity within the same fold
        assert tr.isdisjoint(va) and tr.isdisjoint(te) and va.isdisjoint(te), (
            f"[R1] Overlap detected in fold {idx}"
        )

        # R3 – union covers the entire cohort
        assert len(tr | va | te) == len(all_patients), (
            f"[R3] Missing patients in fold {idx}"
        )

    print("\n ✓ R1 and R3 passed for every fold.")

    # ------------------------------------------------------------------ R2
    test_counter = Counter(pid for fold in folds for pid in fold['test'])
    duplicates = [pid for pid, cnt in test_counter.items() if cnt != 1]

    assert not duplicates, (
        f"[R2] Patients appearing in test 0 or >1 times: {duplicates}"
    )
    print("✓ R2 passed – each patient appears in *test* exactly once.")

    # ------------------------------------------------------------------ R4 (informative)
    def test_load(ids: List[str]) -> int:
        """Total number of measurements for this group of patients."""
        return sum(patient_measurements[pid] for pid in ids)

    loads = [test_load(f['test']) for f in folds]
    mean_load = sum(loads) / len(loads)
    pct_dev = [100 * (l - mean_load) / mean_load for l in loads]

    print("\nTest-set load per fold (number of measurements):", loads)
    print("Deviation from mean (%)                      :", [round(p, 2) for p in pct_dev])

    if any(abs(p) > tolerance_pct for p in pct_dev):
        print(f"⚠️  [R4] Warning: at least one fold deviates by more than {tolerance_pct}% "
              "from the mean test load.")
    else:
        print(f"✓ R4 passed – test loads within ±{tolerance_pct}% of the mean.")

    print(f"\nAll mandatory validation checks for horizon {horizon} completed successfully.")

def save_folds_as_parquet(df_windows: pd.DataFrame, folds: List[Fold], out_dir: str) -> None:
    """
    Save every split of every fold into its own Parquet file.

    Parameters
    ----------
    df_windows : pd.DataFrame
        Full set of windows. Must include column 'patient_id'.

    folds : list[Fold]
        Output of make_folds_greedy_splits or equivalent.

    out_dir : str
        Root directory where `fold_0`, `fold_1`, ... will be created.

    Notes
    -----
    Folder layout created::

        out_dir/
            fold_0/
                train.parquet
                val.parquet
                test.parquet
            fold_1/
                ...
    """
    os.makedirs(out_dir, exist_ok=True)

    for i, fold in enumerate(folds):
        fold_dir = os.path.join(out_dir, f"fold_{i}")
        os.makedirs(fold_dir, exist_ok=True)

        for split in ("train", "val", "test"):
            ids = set(fold[split])          # cast to set for fast lookup
            df_split = df_windows[df_windows["patient_id"].isin(ids)]

            file_path = os.path.join(fold_dir, f"{split}.parquet")
            df_split.to_parquet(file_path, index=False)

        print(f"✓ Fold {i} saved to {fold_dir}")

    print(f"\nAll {len(folds)} folds written under: {out_dir}\n")
    
def tag_folds_and_save(
            df_windows: pd.DataFrame,
            folds: List[Fold],
            output_path: str,
            filename: str = "windows_with_folds_horizon_{horizon}.parquet",
            horizon: int = 2,
            column_prefix: str = "fold") -> None:
    
    """
    Add one column per fold (fold0 … fold{k-1}) indicating whether each window
    belongs to 'train', 'val' or 'test' in that fold, then write a single
    Parquet file.

    Parameters
    ----------
    df_windows : pd.DataFrame
        Original window table (must include column 'patient_id').

    folds : list[Fold]
        Output from make_folds_greedy_splits (patient IDs per split).

    output_path : str
        Destination filename, e.g. 'windows_with_folds.parquet'.

    column_prefix : str, default "fold"
        Prefix for the new columns: fold0, fold1, …

    parquet_kwargs : dict, optional
        Extra arguments passed to DataFrame.to_parquet.
    """    
    df_tagged = df_windows.copy()

    for idx, fold in enumerate(folds):
        # Build a mapping patient_id -> split label for this fold
        label_map = {pid: "train" for pid in fold["train"]}
        label_map.update({pid: "val"  for pid in fold["val"]})
        label_map.update({pid: "test" for pid in fold["test"]})

        col_name = f"{column_prefix}{idx}"
        df_tagged[col_name] = df_tagged["patient_id"].map(label_map).astype("category")

    if not os.path.exists(output_path):
        os.makedirs(output_path)
    
    df_tagged.to_parquet(f'{output_path}{filename}_horizon_{horizon}.parquet', index=False)
    print(f"\n ✓ File {filename}_horizon_{horizon} written: {output_path}  "
          f"({len(df_windows):,} rows, {len(folds)} fold-columns)")
    
    return df_tagged

In [3]:
path_read_data = 'c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/windows/no_split_2025/'
path_to_save_folds = 'c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/all_folds/'
horizon_list = [2, 4, 6, 8, 12, 16]
number_of_folds = 5
validation_fraction = 0.1

for horizon in horizon_list:
    print(f"\nReading data file from {path_read_data} for horizon {horizon}...")
    df_windows = pd.read_parquet(f"{path_read_data}windows_horizon_{horizon}.parquet")
    unique_patient_ids = df_windows['patient_id'].unique()
    num_patients = len(unique_patient_ids)
    
    # Count the number of windows per patient and convert to a dictionary
    patient_number_windows_dict = (
        df_windows
        .groupby("patient_id")   # group all rows that share the same ID
        .size()                  # count rows per group
        .astype(int)             # ensure Python int, not numpy int64
        .to_dict()               # ==> { 'ID_1': n1, 'ID_2': n2, ... }
    )
    
    # Create folds using the greedy split function
    folds_split = make_folds_greedy_splits(patient_number_windows_dict, k=number_of_folds, val_frac=validation_fraction, seed=42)
    
    # Print statistics of the folds created
    print(f'\n ------- Statistics of folds created for horizon: {horizon}-------')
    for i, fold in enumerate(folds_split):
        print(f"\n---- Fold {i + 1}: -----")
        print("-- Totals --")
        print(f"  Train: {len(fold['train'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['train'])}")
        print(f"  Val:   {len(fold['val'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['val'])}") 
        print(f"  Test:  {len(fold['test'])} patients. Total windows: {sum(patient_number_windows_dict[pid] for pid in fold['test'])}")
        
        print("\n -- Percentages --")
        print(f"  Train: {(len(fold['train'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['train']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
        print(f"  Val:   {(len(fold['val'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['val']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
        print(f"  Test:  {(len(fold['test'])/num_patients*100):.2f} percentage of patients. Percentage of windows: {(sum(patient_number_windows_dict[pid] for pid in fold['test']) / sum(patient_number_windows_dict.values()) * 100):.2f}%")
    
    print(f'\nEnd of statistics for folds created for horizon: {horizon}')
    
    # Validate the folds created
    validate_folds(folds=folds_split, patient_measurements=patient_number_windows_dict, horizon=horizon)
    
    # save_folds_as_parquet(df_windows, folds_split, "c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/")
    
    # Save the folds as Parquet files
    df_all = tag_folds_and_save(df_windows, folds_split, f"c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/folds/all_folds/", 'windows_with_folds', horizon=horizon, column_prefix='fold_')


Reading data file from c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/windows/no_split_2025/ for horizon 2...

 ------- Statistics of folds created for horizon: 2-------

---- Fold 1: -----
-- Totals --
  Train: 498 patients. Total windows: 14401748
  Val:   55 patients. Total windows: 1628506
  Test:  138 patients. Total windows: 4007087

 -- Percentages --
  Train: 72.07 percentage of patients. Percentage of windows: 71.87%
  Val:   7.96 percentage of patients. Percentage of windows: 8.13%
  Test:  19.97 percentage of patients. Percentage of windows: 20.00%

---- Fold 2: -----
-- Totals --
  Train: 498 patients. Total windows: 14333965
  Val:   55 patients. Total windows: 1696467
  Test:  138 patients. Total windows: 4006909

 -- Percentages --
  Train: 72.07 percentage of patients. Percentage of windows: 71.54%
  Val:   7.96 percentage of patients. Percentage of windows: 8.47%
  Test:  19.97 percentage of patients. Percentage of windows: 20.00%

---- Fold 3: -----
-- Totals --
  Train: 497 pa

In [4]:
dg

NameError: name 'dg' is not defined

In [ ]:
df_fold_0_train = df_all[df_all['fold1'] == 'train']
df_fold_0_val = df_all[df_all['fold1'] == 'val']
df_fold_0_test = df_all[df_all['fold1'] == 'test']

In [ ]:
df_fold_0_val